# RG3 only: gas and shortshot combined detection
Normal566 + gas17 + shortshot8 = 591 RH shots. Binary defect detection, not defect subtype classification.
Refit RF baseline and RF+ExtraTrees derived ensembles on both defect types. Separate unchanged gas-only RF evaluated on all shortshots to distinguish training modes.
Outer observation-block evaluation; train-only inner threshold selection. All eight shortshots are evaluated without training on their own observation block.
Exploratory repeated-data comparison; no LH merging or timestamp predictors.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
setup=''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0]
setup='\n'.join("d=d.loc[d['product'].eq('RG3')].reset_index(drop=True)" if line.startswith('d=d.loc') else "assert len(d)==591 and y.sum()==25" if line.startswith('assert len(d)==583') else line for line in setup.splitlines())
exec(setup)
kind=np.where(d.PassOrFail.eq(0),'normal',np.where(d.Reason.str.contains('가스',na=False),'gas','shortshot'))
assert int((kind=='gas').sum())==17 and int((kind=='shortshot').sum())==8
def ratio(a,b):
    return a/np.where(np.abs(b)>1e-6,b,np.nan)
F=pd.DataFrame(index=d.index)
F['switch_pressure_fraction']=ratio(d.Max_Switch_Over_Pressure,d.Max_Injection_Pressure)
F['back_pressure_peak_fraction']=ratio(d.Max_Back_Pressure,d.Average_Back_Pressure)
F['pressure_per_speed']=ratio(d.Max_Injection_Pressure,d.Max_Injection_Speed)
F['fill_fraction']=ratio(d.Filling_Time,d.Injection_Time)
F['cushion_fraction']=ratio(d.Cushion_Position,d.Plasticizing_Position)
F['pressure_time_proxy']=d.Max_Injection_Pressure*d.Filling_Time
F['plasticizing_cycle_fraction']=ratio(d.Plasticizing_Time,d.Cycle_Time)
F['cycle_unaccounted_time']=d.Cycle_Time-d.Injection_Time-d.Plasticizing_Time-d.Clamp_Close_Time
F['barrel_rear_front_gap']=d.Barrel_Temperature_2-d.Barrel_Temperature_5
F['hopper_barrel_gap']=d.Hopper_Temperature-d.Barrel_Temperature_5
F['mold_side_gap']=d.Mold_Temperature_3-d.Mold_Temperature_4
F['thermal_hopper_barrel_shake_ratio']=ratio(d.Hopper_Temperature__past10_std,d.Barrel_Temperature_5__past10_std)
# Departures relative to each sensor's own preceding 10-shot variability.
for sensor in ['Cushion_Position','Max_Injection_Pressure','Average_Back_Pressure','Barrel_Temperature_2']:
    F[sensor+'_relative_departure']=ratio(d[sensor+'__dev10'],d[sensor+'__past10_std'])
F['pressure_cushion_departure_product']=F.Max_Injection_Pressure_relative_departure*F.Cushion_Position_relative_departure
F['pressure_back_departure_difference']=F.Max_Injection_Pressure_relative_departure-F.Average_Back_Pressure_relative_departure
F=F.replace([np.inf,-np.inf],np.nan)
views={'transmission6':F.columns[:6].tolist(),'recovery_thermal6':F.columns[6:12].tolist(),'coupling6':F.columns[12:].tolist(),'combined18':F.columns.tolist()}
memo={}
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in memo:return memo[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    out={}
    for name,cols in views.items():
        if len(np.unique(y[ref]))<2:
            out[name+'_mean']=out[name+'_max']=np.zeros(len(q));continue
        s=[]
        for cls in [RandomForestClassifier,ExtraTreesClassifier]:
            m=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),cls(n_estimators=100,max_depth=5,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=2))
            s.append(m.fit(F.iloc[ref][cols],y[ref]).predict_proba(F.iloc[q][cols])[:,1])
        out[name+'_mean']=np.mean(s,axis=0)
        out[name+'_max']=np.max(s,axis=0)
    out['baseline_refit']=pair_predict(ref,q).max(axis=1)
    memo[key]=out
    return out
records=[];audit=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b))
        ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q);inner={n:np.full(len(ref),np.nan) for n in outer}
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if not (~mask).any():continue
            p=predict(ref[~mask],ref[mask])
            for n in inner:inner[n][mask]=p[n]
        for name,score in outer.items():
            cal=inner[name];valid=np.isfinite(cal);norm=cal[valid&(y[ref]==0)]
            cut=float(np.quantile(norm,.99,method='higher')) if len(norm) else 1.
            chosen=cut;best=-1.
            for t in np.unique(np.r_[cal[valid],1.]):
                pr=cal[valid]>t;yy=y[ref][valid]
                fp=int(np.sum(pr&(yy==0)));tp=int(np.sum(pr&(yy==1)));fn=int(np.sum(~pr&(yy==1)))
                if fp>int(np.floor(.02*np.sum(yy==0))):continue
                f=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
                if f>best or (f==best and t>chosen):best=f;chosen=float(t)
            for rule,t in [('normal99',cut),('inner_f1_fp2pct',chosen)]:
                for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,kind=kind[i],actual=int(y[i]),score=float(score[k]),threshold=t,prediction=int(score[k]>t)))
                audit.append(dict(scheme=scheme,block=b,model=name,rule=rule,train=len(ref),train_gas=int(y[ref].sum()),threshold=t,inner_best_F1=best))
        print('Finished',scheme,b,flush=True)
pp=pd.DataFrame(records)
base=pp.loc[pp.model.eq('baseline_refit')&pp.rule.eq('normal99'),['scheme','TimeStamp','prediction']].rename(columns={'prediction':'predicted_train_fpr1'})
rows=[]
for (scheme,name,rule),g in pp.groupby(['scheme','model','rule'],sort=False):
    z=g.merge(base,on=['scheme','TimeStamp'],validate='one_to_one')
    for combine in ['new_only','OR_RF','AND_RF']:
        pred=z.prediction.astype(bool)
        if combine=='OR_RF':pred=pred|z.predicted_train_fpr1
        if combine=='AND_RF':pred=pred&z.predicted_train_fpr1
        tn,fp,fn,tp=confusion_matrix(z.actual,pred,labels=[0,1]).ravel()
        rows.append(dict(scheme=scheme,model=name,rule=rule,combine=combine,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,AP=average_precision_score(z.actual,z.score),shots=len(z),gas_TP=int(np.sum(pred&(z.kind=='gas'))),gas_FN=int(np.sum(~pred&(z.kind=='gas'))),shortshot_TP=int(np.sum(pred&(z.kind=='shortshot'))),shortshot_FN=int(np.sum(~pred&(z.kind=='shortshot')))))
metrics=pd.DataFrame(rows)
F.assign(TimeStamp=d.TimeStamp).to_csv(tables/'rg3_gas_shortshot_joint_features.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_gas_shortshot_joint_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_gas_shortshot_joint_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_gas_shortshot_joint_fold_audit.csv',index=False,encoding='utf-8-sig')
for scheme in metrics.scheme.unique():
    print('\n',scheme)
    print(metrics.loc[metrics.scheme.eq(scheme)].sort_values(['F1','FP'],ascending=[False,True]).head(12).to_string(index=False))

# Separate question: can the unchanged gas-only RF ensemble also flag shortshots?
gas_records=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b))
        ref=np.flatnonzero((~groups.eq(b) if scheme=='leave_one_block_out' else groups.isin(blocks[:j]))&(kind!='shortshot'))
        if len(np.unique(y[ref]))<2:continue
        score=pair_predict(ref,q).max(axis=1)
        cal=np.full(len(ref),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():cal[mask]=pair_predict(ref[~mask],ref[mask]).max(axis=1)
        norm=cal[(y[ref]==0)&np.isfinite(cal)]
        cut=float(np.quantile(norm,.99,method='higher')) if len(norm) else 1.
        for k,i in enumerate(q):gas_records.append(dict(scheme=scheme,TimeStamp=d.TimeStamp.iloc[i],block=b,kind=kind[i],actual=int(y[i]),score=float(score[k]),threshold=cut,prediction=int(score[k]>cut)))
        print('Finished gas-only training / joint evaluation',scheme,b,flush=True)
gp=pd.DataFrame(gas_records);gr=[]
for scheme,z in gp.groupby('scheme',sort=False):
    pr=z.prediction.astype(bool)
    tn,fp,fn,tp=confusion_matrix(z.actual,pr,labels=[0,1]).ravel()
    gr.append(dict(scheme=scheme,model='unchanged_gas_only_RF',TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),gas_TP=int(np.sum(pr&(z.kind=='gas'))),gas_FN=int(np.sum(~pr&(z.kind=='gas'))),shortshot_TP=int(np.sum(pr&(z.kind=='shortshot'))),shortshot_FN=int(np.sum(~pr&(z.kind=='shortshot')))))
gp.to_csv(tables/'rg3_gas_shortshot_gas_only_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(gr).to_csv(tables/'rg3_gas_shortshot_gas_only_metrics.csv',index=False,encoding='utf-8-sig')
# Verify old normal/gas predictions are exactly preserved in gas-only mode.
old=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
old=old.loc[old.model.eq('max_union'),['scheme','TimeStamp','score','predicted_train_fpr1']]
check=gp.loc[gp.kind.ne('shortshot')].merge(old,on=['scheme','TimeStamp'],validate='one_to_one',suffixes=('_new','_old'))
assert len(check)==1023 and np.allclose(check.score_new,check.score_old)
assert (check.prediction.astype(bool)==check.predicted_train_fpr1).all()
cases=pp.loc[pp.kind.eq('shortshot')].pivot_table(index=['scheme','TimeStamp','block'],columns=['model','rule'],values='prediction',aggfunc='first')
cases.to_csv(tables/'rg3_gas_shortshot_all8_cases.csv',encoding='utf-8-sig')
print('Preserved original gas-only predictions:',len(check))
print(pd.DataFrame(gr).to_string(index=False))


In [ ]:
from pathlib import Path
import pandas as pd
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
g=pd.read_csv(tables/'rg3_gas_shortshot_gas_only_predictions.csv')
p=pd.read_csv(tables/'rg3_gas_shortshot_joint_predictions.csv')
cases=g.loc[g.scheme.eq('leave_one_block_out')&g.kind.eq('shortshot'),['TimeStamp','block','prediction','score','threshold']].rename(columns={'prediction':'gas_only_detected','score':'gas_only_score','threshold':'gas_only_threshold'})
for name,label in [('baseline_refit','joint_RF_detected'),('coupling6_max','joint_coupling_detected')]:
    q=p.loc[p.scheme.eq('leave_one_block_out')&p.kind.eq('shortshot')&p.model.eq(name)&p.rule.eq('normal99'),['TimeStamp','prediction']].rename(columns={'prediction':label})
    cases=cases.merge(q,on='TimeStamp',validate='one_to_one')
assert len(cases)==8
cases.to_csv(tables/'rg3_gas_shortshot_all8_readable.csv',index=False,encoding='utf-8-sig')
print(cases[['TimeStamp','gas_only_detected','joint_RF_detected','joint_coupling_detected']].to_string(index=False))


## Executed results
Whole evaluation: gas-only RF gas4/17, shortshot0/8, FP6, FN21, TN560, F1=0.2286. Joint RF gas3/17, shortshot0/8, FP11, FN22, TN555, F1=0.1538.
All tested derived combinations detect zero shortshots with the tested thresholds.
Forward evaluation contains gas13, shortshot7, normal427: gas-only RF gas2, shortshot0, FP4, F1=0.1538; joint RF gas1, shortshot0, FP7, F1=0.0714.
Verified that original 1023 normal/gas scores and decisions across both schemes are unchanged in gas-only training mode.
Negative findings apply only to these features/models/thresholds. No defect-subtype classifier was trained. See reports/rg3_gas_shortshot_joint_comparison.md.